> Adapted from *Dive into Deep Learning*, section [The Encoder--Decoder Architecture](https://d2l.ai/chapter_recurrent-modern/encoder-decoder.html). Not affiliated with or endorsed by the book's authors.
> Source, authors, licenses and changes: [Attribution and changes](#Attribution-and-changes), at the end of this notebook.

Previous: [Section 10.5: Machine Translation and the Dataset](10.05_machine-translation-and-dataset.ipynb) | Next: [Section 10.7: Sequence-to-Sequence Learning for Machine Translation](10.07_seq2seq.ipynb)

# The Encoder--Decoder Architecture

In general sequence-to-sequence problems
like machine translation
([Section 10.5](https://d2l.ai/chapter_recurrent-modern/machine-translation-and-dataset.html#sec-machine-translation)),
inputs and outputs are of varying lengths
that are unaligned.
The standard approach to handling this sort of data
is to design an *encoder--decoder* architecture ([Fig. 10.6.1](https://d2l.ai/chapter_recurrent-modern/encoder-decoder.html#fig-encoder-decoder))
consisting of two major components:
an *encoder* that takes a variable-length sequence as input,
and a *decoder* that acts as a conditional language model,
taking in the encoded input
and the leftwards context of the target sequence
and predicting the subsequent token in the target sequence.


The encoder--decoder architecture.

Let's take machine translation from English to French as an example.
Given an input sequence in English:
"They", "are", "watching", ".",
this encoder--decoder architecture
first encodes the variable-length input into a state,
then decodes the state
to generate the translated sequence,
token by token, as output:
"Ils", "regardent", ".".
Since the encoder--decoder architecture
forms the basis of different sequence-to-sequence models
in subsequent sections,
this section will convert this architecture
into an interface that will be implemented later.

In [ ]:
from divelet_ml import nn

import divelet_ml as d2l

## Encoder

In the encoder interface,
we just specify that
the encoder takes variable-length sequences as input `X`.
The implementation will be provided
by any model that inherits this base `Encoder` class.

In [ ]:
class Encoder(nn.Module):  #@save
    """The base encoder interface for the encoder--decoder architecture."""
    def __init__(self):
        super().__init__()

    # Later there can be additional arguments (e.g., length excluding padding)
    def forward(self, X, *args):
        raise NotImplementedError

## Decoder

In the following decoder interface,
we add an additional `init_state` method
to convert the encoder output (`enc_all_outputs`)
into the encoded state.
Note that this step
may require extra inputs,
such as the valid length of the input,
which was explained
in [Section 10.5](https://d2l.ai/chapter_recurrent-modern/machine-translation-and-dataset.html#sec-machine-translation).
To generate a variable-length sequence token by token,
every time the decoder may map an input
(e.g., the generated token at the previous time step)
and the encoded state
into an output token at the current time step.

In [ ]:
class Decoder(nn.Module):  #@save
    """The base decoder interface for the encoder--decoder architecture."""
    def __init__(self):
        super().__init__()

    # Later there can be additional arguments (e.g., length excluding padding)
    def init_state(self, enc_all_outputs, *args):
        raise NotImplementedError

    def forward(self, X, state):
        raise NotImplementedError

## Putting the Encoder and Decoder Together

In the forward propagation,
the output of the encoder
is used to produce the encoded state,
and this state will be further used
by the decoder as one of its input.

In [ ]:
class EncoderDecoder(d2l.Classifier):  #@save
    """The base class for the encoder--decoder architecture."""
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, enc_X, dec_X, *args):
        enc_all_outputs = self.encoder(enc_X, *args)
        dec_state = self.decoder.init_state(enc_all_outputs, *args)
        # Return decoder output only
        return self.decoder(dec_X, dec_state)[0]

In the next section,
we will see how to apply RNNs to design
sequence-to-sequence models based on
this encoder--decoder architecture.


## Summary

Encoder-decoder architectures
can handle inputs and outputs
that both consist of variable-length sequences
and thus are suitable for sequence-to-sequence problems
such as machine translation.
The encoder takes a variable-length sequence as input
and transforms it into a state with a fixed shape.
The decoder maps the encoded state of a fixed shape
to a variable-length sequence.


## Exercises

1. Suppose that we use neural networks to implement the encoder--decoder architecture. Do the encoder and the decoder have to be the same type of neural network?
1. Besides machine translation, can you think of another application where the encoder--decoder architecture can be applied?

Discussions of the original section: [book's forum](https://discuss.d2l.ai/t/1061)

Previous: [Section 10.5: Machine Translation and the Dataset](10.05_machine-translation-and-dataset.ipynb) | Next: [Section 10.7: Sequence-to-Sequence Learning for Machine Translation](10.07_seq2seq.ipynb)

## Attribution and changes

Adapted from *Dive into Deep Learning*, section
[The Encoder--Decoder Architecture](https://d2l.ai/chapter_recurrent-modern/encoder-decoder.html)
(source: [`chapter_recurrent-modern/encoder-decoder.md`](https://github.com/d2l-ai/d2l-en/blob/23d7a5aecceee57d1292c56e90cce307f183bb0a/chapter_recurrent-modern/encoder-decoder.md)
at commit `23d7a5aeccee`), by Aston Zhang, Zachary C. Lipton, Mu Li, Alexander J. Smola and contributors.
Text and figures: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). Code: adapted from the book's sample code, MIT-0.
Not affiliated with or endorsed by the book's authors.

**Changes from the original** (PyTorch edition):
- Code uses NumPy instead of PyTorch, so it runs in JupyterLite (Pyodide and xeus-python).
- `d2l` is divelet-ml's package (`import divelet_ml as d2l`) instead of the book's `d2l.torch`. `%matplotlib inline` is removed: both JupyterLite kernels display plots inline.
- `nn` is divelet-ml's `divelet_ml.nn`. The `Encoder`, `Decoder` and `EncoderDecoder` classes of this section are in divelet-ml as `d2l.Encoder`, `d2l.Decoder` and `d2l.EncoderDecoder`.
- Equations keep the book's numbers. Cross-references to other sections link to the book's site, and citations are written inline (author and year); the book's bibliography lists them.
- The d2lbook framework tabs for MXNet, TensorFlow and JAX are removed.